Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


Load Dataset


In [4]:
multilingual_df = pd.read_csv("GWSD_Multilingual.tsv", sep="\t")

print("Dataset loaded successfully!")
print("Shape:", multilingual_df.shape)
print(multilingual_df.columns.tolist())

Dataset loaded successfully!
Shape: (2300, 20)
['Unnamed: 0', 'round', 'batch', 'sent_id', 'sentence', 'worker_0', 'worker_1', 'worker_2', 'worker_3', 'worker_4', 'worker_5', 'worker_6', 'worker_7', 'disagree', 'agree', 'neutral', 'guid', 'in_held_out_test', 'sentence_hi', 'sentence_bn']


In [5]:
print("Dataset shape:", multilingual_df.shape)

print("\nColumns:")
print(multilingual_df.columns.tolist())

print("\nFirst 5 rows:")
display(multilingual_df.head())

Dataset shape: (2300, 20)

Columns:
['Unnamed: 0', 'round', 'batch', 'sent_id', 'sentence', 'worker_0', 'worker_1', 'worker_2', 'worker_3', 'worker_4', 'worker_5', 'worker_6', 'worker_7', 'disagree', 'agree', 'neutral', 'guid', 'in_held_out_test', 'sentence_hi', 'sentence_bn']

First 5 rows:


,Unnamed: 0,round,batch,sent_id,sentence,worker_0,worker_1,worker_2,worker_3,worker_4,worker_5,worker_6,worker_7,disagree,agree,neutral,guid,in_held_out_test,sentence_hi,sentence_bn
0,0,1,0,s0,Global warming is a hoax.,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,NaN,NaN,NaN,1_0_s0,False,वैश्विक तापमान एक hax है.,গ্লোবাল গরম একটি হোকস।
1,1,1,0,s1,Alarming levels of sea level rise are predicte...,neutral,neutral,neutral,agrees,agrees,neutral,neutral,agrees,NaN,NaN,NaN,1_0_s1,False,समुद्र स्तर के अलार्म के स्तर अगले दशकों के दौ...,পরবর্তী দশক ধরে ফ্রিলিডিকে হুমকি দেবে নৌবাহিনী...
2,2,1,0,s2,"Over the past several years, the United States...",neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s2,False,"पिछले कई सालों से, अमरीका ने व्यापार वृद्धि मे...","গত কয়েক বছর ধরে, যুক্তরাষ্ট্র ব্যবসায়িক বৃদ্..."
3,3,1,0,s3,Global warming is happening and it will be dan...,agrees,agrees,agrees,agrees,agrees,agrees,agrees,agrees,NaN,NaN,NaN,1_0_s3,False,विश्‍वव्यापी तापमान हो रहा है और यह मानव स्वास...,গ্লোবাল গরমের ঘটনা ঘটেছে এবং এটি মানবাধিকার ও ...
4,4,1,0,s4,Some icebergs are cute.,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s4,False,कुछ कीड़े काट रहे हैं.,কিছু আইসবেরি খুবই সুন্দর।


Create stance labels

In [6]:
score_columns = ["agree", "disagree", "neutral"]

# Remove rows with missing stance scores
labelled_df = multilingual_df.dropna(
    subset=score_columns
).copy()

# Assign the highest-scoring stance
labelled_df["stance"] = labelled_df[
    score_columns
].idxmax(axis=1)

# Convert into readable class names
labelled_df["stance"] = labelled_df["stance"].map({
    "agree": "Favour",
    "disagree": "Against",
    "neutral": "Neutral"
})

print("Labelled dataset shape:", labelled_df.shape)

print("\nClass distribution:")
print(labelled_df["stance"].value_counts())

Labelled dataset shape: (2050, 21)

Class distribution:
stance
Neutral    873
Favour     777
Against    400
Name: count, dtype: int64


Create training and testing datasets
We'll use the original GWSD split:
- Training: 1,850 sentences
- Testing: 200 sentences

In [7]:
# Separate training data
train_df = labelled_df[
    labelled_df["in_held_out_test"] == False
].copy()

# Separate testing data
test_df = labelled_df[
    labelled_df["in_held_out_test"] == True
].copy()

# Reset indexes
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

print("\nTraining class distribution:")
print(train_df["stance"].value_counts())

print("\nTesting class distribution:")
print(test_df["stance"].value_counts())

Training dataset: (1850, 21)
Testing dataset: (200, 21)

Training class distribution:
stance
Neutral    788
Favour     701
Against    361
Name: count, dtype: int64

Testing class distribution:
stance
Neutral    85
Favour     76
Against    39
Name: count, dtype: int64


Verify the split

In [8]:
print("Training records:", len(train_df))
print("Testing records:", len(test_df))

print("\nTraining GUIDs:", train_df["guid"].nunique())
print("Testing GUIDs:", test_df["guid"].nunique())

print("\nAny overlap between train and test?")
print(set(train_df["guid"]).intersection(set(test_df["guid"])))

Training records: 1850
Testing records: 200

Training GUIDs: 1850
Testing GUIDs: 200

Any overlap between train and test?
set()


# Experiment 1: English TF-IDF + Logistic Regression

In [9]:
# Separate English text and labels
X_train_en = train_df["sentence"]
X_test_en = test_df["sentence"]

y_train = train_df["stance"]
y_test = test_df["stance"]

# TF-IDF feature extraction
tfidf_en = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_en_tfidf = tfidf_en.fit_transform(X_train_en)
X_test_en_tfidf = tfidf_en.transform(X_test_en)

print("Training feature shape:", X_train_en_tfidf.shape)
print("Testing feature shape:", X_test_en_tfidf.shape)

Training feature shape: (1850, 10000)
Testing feature shape: (200, 10000)


Train Logistic Regression

In [10]:
# Initialize Logistic Regression
lr_en = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train model
lr_en.fit(X_train_en_tfidf, y_train)

# Make predictions
y_pred_en_lr = lr_en.predict(X_test_en_tfidf)

# Evaluate model
print("English - TF-IDF + Logistic Regression")
print("----------------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_en_lr))

print("Precision:",
      precision_score(y_test, y_pred_en_lr, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_en_lr, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_en_lr, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_lr))

English - TF-IDF + Logistic Regression
----------------------------------------
Accuracy: 0.58
Precision: 0.5761503618646475
Recall: 0.5988290862903866
Macro F1 Score: 0.5842985409652076

Classification Report:
              precision    recall  f1-score   support

     Against       0.55      0.69      0.61        39
      Favour       0.55      0.54      0.55        76
     Neutral       0.62      0.56      0.59        85

    accuracy                           0.58       200
   macro avg       0.58      0.60      0.58       200
weighted avg       0.58      0.58      0.58       200



Hindi TF-IDF + Logistic Regression

In [11]:
# Hindi text and labels
X_train_hi = train_df["sentence_hi"]
X_test_hi = test_df["sentence_hi"]

# TF-IDF feature extraction
tfidf_hi = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_hi_tfidf = tfidf_hi.fit_transform(X_train_hi)
X_test_hi_tfidf = tfidf_hi.transform(X_test_hi)

print("Hindi training feature shape:",
      X_train_hi_tfidf.shape)

print("Hindi testing feature shape:",
      X_test_hi_tfidf.shape)

# Train Logistic Regression
lr_hi = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_hi.fit(X_train_hi_tfidf, y_train)

# Predictions
y_pred_hi_lr = lr_hi.predict(X_test_hi_tfidf)

# Evaluation
print("\nHindi - TF-IDF + Logistic Regression")
print("--------------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_hi_lr))

print("Precision:",
      precision_score(y_test, y_pred_hi_lr, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_hi_lr, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_hi_lr, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_lr))

Hindi training feature shape: (1850, 10000)
Hindi testing feature shape: (200, 10000)

Hindi - TF-IDF + Logistic Regression
--------------------------------------
Accuracy: 0.53
Precision: 0.5165585835033518
Recall: 0.5272353205789738
Macro F1 Score: 0.5203021316647038

Classification Report:
              precision    recall  f1-score   support

     Against       0.43      0.51      0.47        39
      Favour       0.56      0.54      0.55        76
     Neutral       0.56      0.53      0.55        85

    accuracy                           0.53       200
   macro avg       0.52      0.53      0.52       200
weighted avg       0.54      0.53      0.53       200



Bengali TF-IDF + Logistic Regression

In [12]:
# Bengali text
X_train_bn = train_df["sentence_bn"]
X_test_bn = test_df["sentence_bn"]

# TF-IDF feature extraction
tfidf_bn = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_bn_tfidf = tfidf_bn.fit_transform(X_train_bn)
X_test_bn_tfidf = tfidf_bn.transform(X_test_bn)

print("Bengali training feature shape:",
      X_train_bn_tfidf.shape)

print("Bengali testing feature shape:",
      X_test_bn_tfidf.shape)

# Train Logistic Regression
lr_bn = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_bn.fit(X_train_bn_tfidf, y_train)

# Predictions
y_pred_bn_lr = lr_bn.predict(X_test_bn_tfidf)

# Evaluation
print("\nBengali - TF-IDF + Logistic Regression")
print("----------------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_bn_lr))

print("Precision:",
      precision_score(y_test, y_pred_bn_lr, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_bn_lr, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_bn_lr, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_lr))

Bengali training feature shape: (1850, 10000)
Bengali testing feature shape: (200, 10000)

Bengali - TF-IDF + Logistic Regression
----------------------------------------
Accuracy: 0.565
Precision: 0.5423337502605796
Recall: 0.5431319626366067
Macro F1 Score: 0.5420517201211462

Classification Report:
              precision    recall  f1-score   support

     Against       0.42      0.44      0.43        39
      Favour       0.56      0.61      0.58        76
     Neutral       0.64      0.59      0.61        85

    accuracy                           0.56       200
   macro avg       0.54      0.54      0.54       200
weighted avg       0.57      0.56      0.57       200



In [13]:
baseline_results = [
    {
        "Language": "English",
        "Model": "Logistic Regression",
        "Accuracy": 0.580,
        "Macro F1": 0.5843
    },
    {
        "Language": "Hindi",
        "Model": "Logistic Regression",
        "Accuracy": 0.530,
        "Macro F1": 0.5203
    },
    {
        "Language": "Bengali",
        "Model": "Logistic Regression",
        "Accuracy": 0.565,
        "Macro F1": 0.5421
    }
]

print("Previous results saved!")

Previous results saved!


# Experiment 2: TF-IDF + Linear SVM

In [14]:
# Initialize Linear SVM
svm_en = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train model
svm_en.fit(X_train_en_tfidf, y_train)

# Predictions
y_pred_en_svm = svm_en.predict(X_test_en_tfidf)

# Evaluation
print("English - TF-IDF + Linear SVM")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_en_svm))

print("Precision:",
      precision_score(y_test, y_pred_en_svm, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_en_svm, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_en_svm, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_svm))

English - TF-IDF + Linear SVM
--------------------------------
Accuracy: 0.565
Precision: 0.5554823115798726
Recall: 0.5648659733799052
Macro F1 Score: 0.5594347048874139

Classification Report:
              precision    recall  f1-score   support

     Against       0.50      0.56      0.53        39
      Favour       0.58      0.57      0.57        76
     Neutral       0.59      0.56      0.57        85

    accuracy                           0.56       200
   macro avg       0.56      0.56      0.56       200
weighted avg       0.57      0.56      0.57       200



Hindi TF-IDF + Linear SVM

In [15]:
# Initialize Hindi Linear SVM
svm_hi = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train model
svm_hi.fit(X_train_hi_tfidf, y_train)

# Predictions
y_pred_hi_svm = svm_hi.predict(X_test_hi_tfidf)

# Evaluation
print("Hindi - TF-IDF + Linear SVM")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_hi_svm))

print("Precision:",
      precision_score(y_test, y_pred_hi_svm, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_hi_svm, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_hi_svm, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_svm))

Hindi - TF-IDF + Linear SVM
--------------------------------
Accuracy: 0.525
Precision: 0.5112163029525032
Recall: 0.5103662247625095
Macro F1 Score: 0.5105875495097052

Classification Report:
              precision    recall  f1-score   support

     Against       0.45      0.44      0.44        39
      Favour       0.54      0.57      0.55        76
     Neutral       0.55      0.53      0.54        85

    accuracy                           0.53       200
   macro avg       0.51      0.51      0.51       200
weighted avg       0.52      0.53      0.52       200



Bengali TF-IDF + Linear SVM

In [16]:
# Initialize Bengali Linear SVM
svm_bn = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train model
svm_bn.fit(X_train_bn_tfidf, y_train)

# Predictions
y_pred_bn_svm = svm_bn.predict(X_test_bn_tfidf)

# Evaluation
print("Bengali - TF-IDF + Linear SVM")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(y_test, y_pred_bn_svm))

print("Precision:",
      precision_score(y_test, y_pred_bn_svm, average="macro"))

print("Recall:",
      recall_score(y_test, y_pred_bn_svm, average="macro"))

print("Macro F1 Score:",
      f1_score(y_test, y_pred_bn_svm, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_svm))

Bengali - TF-IDF + Linear SVM
--------------------------------
Accuracy: 0.555
Precision: 0.5291233714476182
Recall: 0.521412505623032
Macro F1 Score: 0.5236986344770775

Classification Report:
              precision    recall  f1-score   support

     Against       0.42      0.36      0.39        39
      Favour       0.54      0.61      0.57        76
     Neutral       0.62      0.60      0.61        85

    accuracy                           0.56       200
   macro avg       0.53      0.52      0.52       200
weighted avg       0.55      0.56      0.55       200



In [17]:
baseline_results = [
    {"Language": "English", "Model": "Logistic Regression", "Accuracy": 0.580, "Macro F1": 0.5843},
    {"Language": "Hindi", "Model": "Logistic Regression", "Accuracy": 0.530, "Macro F1": 0.5203},
    {"Language": "Bengali", "Model": "Logistic Regression", "Accuracy": 0.565, "Macro F1": 0.5421},
    {"Language": "English", "Model": "Linear SVM", "Accuracy": 0.565, "Macro F1": 0.5594},
    {"Language": "Hindi", "Model": "Linear SVM", "Accuracy": 0.525, "Macro F1": 0.5106},
    {"Language": "Bengali", "Model": "Linear SVM", "Accuracy": 0.555, "Macro F1": 0.5237}
]

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,Language,Model,Accuracy,Macro F1
0,English,Logistic Regression,0.580,0.5843
1,Hindi,Logistic Regression,0.530,0.5203
2,Bengali,Logistic Regression,0.565,0.5421
3,English,Linear SVM,0.565,0.5594
4,Hindi,Linear SVM,0.525,0.5106
5,Bengali,Linear SVM,0.555,0.5237


# EXPERIMENT-3 FastText + Logistic Regression

In [18]:
!pip install -q gensim

Import libraries and tokenize text

In [19]:
from gensim.models import FastText
from gensim.utils import simple_preprocess

# Prepare sentences for each language
sentences_en = [
    simple_preprocess(str(text), min_len=1)
    for text in train_df["sentence"]
]

sentences_hi = [
    simple_preprocess(str(text), min_len=1)
    for text in train_df["sentence_hi"]
]

sentences_bn = [
    simple_preprocess(str(text), min_len=1)
    for text in train_df["sentence_bn"]
]

print("English sentences:", len(sentences_en))
print("Hindi sentences:", len(sentences_hi))
print("Bengali sentences:", len(sentences_bn))

print("\nEnglish example:", sentences_en[0])
print("Hindi example:", sentences_hi[0])
print("Bengali example:", sentences_bn[0])

English sentences: 1850
Hindi sentences: 1850
Bengali sentences: 1850

English example: ['warmer', 'than', 'normal', 'sea', 'surface', 'temperatures', 'are', 'a', 'key', 'player', 'in', 'the', 'development', 'of', 'hurricanes', 'such', 'as', 'katrina', 'and', 'superstorm', 'sandy', 'which', 'hit', 'the', 'u', 's', 'east', 'coast', 'in']
Hindi example: ['u', 's', 'प', 'र', 'व', 'तट', 'पर', 'u', 's', 'प', 'र', 'व', 'तट', 'म', 'र']
Bengali example: ['ত', 'পম', 'ত', 'র', 'ন', 'ক', 'ত', 'ক', 'ন', 'ক', 'ত', 'ক', 'স', 'গর', 'প', 'থ', 'ব', 'র', 'ত', 'পম', 'ত', 'র', 'ক', 'য', 'ট', 'র', 'ন', 'এব', 'স', 'প', 'র', 'স', 'ট', 'রম', 'স', 'ন', 'ড', 'ज', 'स', 'হ', 'রগ', 'নগ', 'ল', 'র', 'উন', 'নয', 'ন', 'একট', 'গ', 'র', 'ত', 'বপ', 'র', 'ণ', 'খ', 'ল', 'হয', 'য', 'স', 'ল', 'ম', 'র', 'ক', 'ন', 'প', 'র', 'ব', 'ক', 'স', 'টম', 'র', 'দ', 'ক', 'হ', 'মল', 'র']


(Correction): Use Unicode-aware tokenization

In [20]:
!pip install -q regex

import regex as re

def tokenize_multilingual(text):
    text = str(text).lower()
    tokens = re.findall(r'[\p{L}\p{M}\p{N}]+', text)
    return tokens

# Tokenize all three languages correctly
sentences_en = [
    tokenize_multilingual(text) for text in train_df["sentence"]
]

sentences_hi = [
    tokenize_multilingual(text) for text in train_df["sentence_hi"]
]

sentences_bn = [
    tokenize_multilingual(text) for text in train_df["sentence_bn"]
]

print("English example:", sentences_en[0])
print("Hindi example:", sentences_hi[0])
print("Bengali example:", sentences_bn[0])

English example: ['warmer', 'than', 'normal', 'sea', 'surface', 'temperatures', 'are', 'a', 'key', 'player', 'in', 'the', 'development', 'of', 'hurricanes', 'such', 'as', 'katrina', 'and', 'superstorm', 'sandy', 'which', 'hit', 'the', 'u', 's', 'east', 'coast', 'in', '2011']
Hindi example: ['u', 's', 'पूर्वी', 'तट', 'पर', 'u', 's', 'पूर्वी', 'तट', 'मारा']
Bengali example: ['তাপমাত্রা', 'নাক্তিক', 'নাক্তিক', 'সাগর', 'পৃথিবীর', 'তাপমাত্রা', 'ক্যাট্রিনা', 'এবং', 'সুপার', 'স্টোরম', 'সান্ডি', 'जैसे', 'হুরগানগুলির', 'উন্নয়নে', 'একটি', 'গুরুত্বপূর্ণ', 'খেলা', 'হয়', 'যা', '২০১১', 'সালে', 'মার্কিন', 'পূর্ব', 'কাস্টমের', 'দিকে', 'হামলার']


Train three separate FastText models

In [21]:
# Train English FastText
fasttext_en = FastText(
    sentences=sentences_en,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

# Train Hindi FastText
fasttext_hi = FastText(
    sentences=sentences_hi,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

# Train Bengali FastText
fasttext_bn = FastText(
    sentences=sentences_bn,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

print("All three FastText models trained successfully!")

All three FastText models trained successfully!


Convert sentences into numerical vectors

In [22]:
import numpy as np

def get_sentence_vectors(sentences, model):
    vectors = []

    for sentence in sentences:
        word_vectors = [model.wv[word] for word in sentence]

        if len(word_vectors) > 0:
            sentence_vector = np.mean(word_vectors, axis=0)
        else:
            sentence_vector = np.zeros(model.vector_size)

        vectors.append(sentence_vector)

    return np.array(vectors)


# Training vectors
X_train_en_fasttext = get_sentence_vectors(sentences_en, fasttext_en)
X_train_hi_fasttext = get_sentence_vectors(sentences_hi, fasttext_hi)
X_train_bn_fasttext = get_sentence_vectors(sentences_bn, fasttext_bn)

print("English shape:", X_train_en_fasttext.shape)
print("Hindi shape:", X_train_hi_fasttext.shape)
print("Bengali shape:", X_train_bn_fasttext.shape)

English shape: (1850, 100)
Hindi shape: (1850, 100)
Bengali shape: (1850, 100)


Convert test sentences into FastText vectors

In [23]:
# Tokenize test sentences
test_sentences_en = [
    tokenize_multilingual(text) for text in test_df["sentence"]
]

test_sentences_hi = [
    tokenize_multilingual(text) for text in test_df["sentence_hi"]
]

test_sentences_bn = [
    tokenize_multilingual(text) for text in test_df["sentence_bn"]
]

# Convert test sentences into vectors
X_test_en_fasttext = get_sentence_vectors(test_sentences_en, fasttext_en)
X_test_hi_fasttext = get_sentence_vectors(test_sentences_hi, fasttext_hi)
X_test_bn_fasttext = get_sentence_vectors(test_sentences_bn, fasttext_bn)

print("English test shape:", X_test_en_fasttext.shape)
print("Hindi test shape:", X_test_hi_fasttext.shape)
print("Bengali test shape:", X_test_bn_fasttext.shape)

English test shape: (200, 100)
Hindi test shape: (200, 100)
Bengali test shape: (200, 100)


English FastText + Logistic Regression

In [24]:
# Train Logistic Regression on English FastText vectors
lr_en_fasttext = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_en_fasttext.fit(X_train_en_fasttext, y_train)

# Predictions
y_pred_en_fasttext = lr_en_fasttext.predict(X_test_en_fasttext)

# Evaluation
print("English - FastText + Logistic Regression")
print("-----------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_en_fasttext))
print("Precision:", precision_score(y_test, y_pred_en_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_en_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_en_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_fasttext))

English - FastText + Logistic Regression
-----------------------------------------
Accuracy: 0.56
Precision: 0.5561958874458874
Recall: 0.5775885792913656
Macro F1 Score: 0.5535339188277173

Classification Report:
              precision    recall  f1-score   support

     Against       0.41      0.67      0.50        39
      Favour       0.59      0.51      0.55        76
     Neutral       0.67      0.55      0.61        85

    accuracy                           0.56       200
   macro avg       0.56      0.58      0.55       200
weighted avg       0.59      0.56      0.56       200



Hindi – FastText + Logistic Regression.

In [25]:
lr_hi_fasttext = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_hi_fasttext.fit(X_train_hi_fasttext, y_train)

y_pred_hi_fasttext = lr_hi_fasttext.predict(X_test_hi_fasttext)

print("Hindi - FastText + Logistic Regression")
print("--------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_hi_fasttext))
print("Precision:", precision_score(y_test, y_pred_hi_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_hi_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_hi_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_fasttext))

Hindi - FastText + Logistic Regression
--------------------------------------
Accuracy: 0.53
Precision: 0.5344827586206896
Recall: 0.5568270223068984
Macro F1 Score: 0.5257999049043826

Classification Report:
              precision    recall  f1-score   support

     Against       0.38      0.69      0.49        39
      Favour       0.60      0.46      0.52        76
     Neutral       0.62      0.52      0.56        85

    accuracy                           0.53       200
   macro avg       0.53      0.56      0.53       200
weighted avg       0.57      0.53      0.53       200



Bengali – FastText + Logistic Regression

In [26]:
lr_bn_fasttext = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_bn_fasttext.fit(X_train_bn_fasttext, y_train)

y_pred_bn_fasttext = lr_bn_fasttext.predict(X_test_bn_fasttext)

print("Bengali - FastText + Logistic Regression")
print("----------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_bn_fasttext))
print("Precision:", precision_score(y_test, y_pred_bn_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_bn_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_bn_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_fasttext))

Bengali - FastText + Logistic Regression
----------------------------------------
Accuracy: 0.605
Precision: 0.5947087309500071
Recall: 0.6128826969384246
Macro F1 Score: 0.5946565760807246

Classification Report:
              precision    recall  f1-score   support

     Against       0.46      0.67      0.55        39
      Favour       0.66      0.51      0.58        76
     Neutral       0.66      0.66      0.66        85

    accuracy                           0.60       200
   macro avg       0.59      0.61      0.59       200
weighted avg       0.62      0.60      0.61       200



# EXPERIMENT 4- FastText + Linear SVM

English FastText + Linear SVM

In [27]:
svm_en_fasttext = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_en_fasttext.fit(X_train_en_fasttext, y_train)

y_pred_en_svm_fasttext = svm_en_fasttext.predict(
    X_test_en_fasttext
)

print("English - FastText + Linear SVM")
print("--------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_en_svm_fasttext))
print("Precision:", precision_score(y_test, y_pred_en_svm_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_en_svm_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_en_svm_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_svm_fasttext))

English - FastText + Linear SVM
--------------------------------
Accuracy: 0.615
Precision: 0.5999728576011054
Recall: 0.6101002884284618
Macro F1 Score: 0.6038062636265461

Classification Report:
              precision    recall  f1-score   support

     Against       0.51      0.59      0.55        39
      Favour       0.61      0.61      0.61        76
     Neutral       0.68      0.64      0.66        85

    accuracy                           0.61       200
   macro avg       0.60      0.61      0.60       200
weighted avg       0.62      0.61      0.62       200



Hindi – FastText + Linear SVM

In [28]:
svm_hi_fasttext = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_hi_fasttext.fit(X_train_hi_fasttext, y_train)

y_pred_hi_svm_fasttext = svm_hi_fasttext.predict(
    X_test_hi_fasttext
)

print("Hindi - FastText + Linear SVM")
print("------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_hi_svm_fasttext))
print("Precision:", precision_score(y_test, y_pred_hi_svm_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_hi_svm_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_hi_svm_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_svm_fasttext))

Hindi - FastText + Linear SVM
------------------------------
Accuracy: 0.59
Precision: 0.5742507190771549
Recall: 0.5747585403932153
Macro F1 Score: 0.5703411089276917

Classification Report:
              precision    recall  f1-score   support

     Against       0.43      0.51      0.47        39
      Favour       0.69      0.55      0.61        76
     Neutral       0.61      0.66      0.63        85

    accuracy                           0.59       200
   macro avg       0.57      0.57      0.57       200
weighted avg       0.60      0.59      0.59       200



Bengali – FastText + Linear SVM

In [29]:
svm_bn_fasttext = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_bn_fasttext.fit(X_train_bn_fasttext, y_train)

y_pred_bn_svm_fasttext = svm_bn_fasttext.predict(
    X_test_bn_fasttext
)

print("Bengali - FastText + Linear SVM")
print("--------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_bn_svm_fasttext))
print("Precision:", precision_score(y_test, y_pred_bn_svm_fasttext, average="macro"))
print("Recall:", recall_score(y_test, y_pred_bn_svm_fasttext, average="macro"))
print("Macro F1 Score:", f1_score(y_test, y_pred_bn_svm_fasttext, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_svm_fasttext))

Bengali - FastText + Linear SVM
--------------------------------
Accuracy: 0.635
Precision: 0.6172161172161172
Recall: 0.6160712868143209
Macro F1 Score: 0.6160272376025802

Classification Report:
              precision    recall  f1-score   support

     Against       0.54      0.54      0.54        39
      Favour       0.64      0.59      0.62        76
     Neutral       0.67      0.72      0.69        85

    accuracy                           0.64       200
   macro avg       0.62      0.62      0.62       200
weighted avg       0.63      0.64      0.63       200



In [30]:
all_results = [
    # TF-IDF
    {"Language": "English", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.580, "Macro F1": 0.5843},
    {"Language": "English", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.565, "Macro F1": 0.5594},
    {"Language": "Hindi", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.530, "Macro F1": 0.5203},
    {"Language": "Hindi", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.525, "Macro F1": 0.5106},
    {"Language": "Bengali", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.565, "Macro F1": 0.5421},
    {"Language": "Bengali", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.555, "Macro F1": 0.5237},

    # FastText
    {"Language": "English", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.560, "Macro F1": 0.5535},
    {"Language": "English", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.615, "Macro F1": 0.6038},
    {"Language": "Hindi", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.530, "Macro F1": 0.5258},
    {"Language": "Hindi", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.590, "Macro F1": 0.5703},
    {"Language": "Bengali", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.605, "Macro F1": 0.5947},
    {"Language": "Bengali", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.635, "Macro F1": 0.6160}
]

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

results_df

,Language,Feature,Model,Accuracy,Macro F1
0,Bengali,FastText,Linear SVM,0.635,0.6160
1,English,FastText,Linear SVM,0.615,0.6038
2,Bengali,FastText,Logistic Regression,0.605,0.5947
3,English,TF-IDF,Logistic Regression,0.580,0.5843
4,Hindi,FastText,Linear SVM,0.590,0.5703
5,English,TF-IDF,Linear SVM,0.565,0.5594
6,English,FastText,Logistic Regression,0.560,0.5535
7,Bengali,TF-IDF,Logistic Regression,0.565,0.5421
8,Hindi,FastText,Logistic Regression,0.530,0.5258
9,Bengali,TF-IDF,Linear SVM,0.555,0.5237
